In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

placer_api_key = os.getenv('PLACER_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if not placer_api_key:
    raise ValueError("PLACER_API_KEY not found in .env file")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env file")

print("✓ API keys loaded successfully")

## Competitor Analysis - Car Wash Services

**Target POI:** 1313 Woodbury Ave, Portsmouth, NH  
**Coordinates:** latitude 43.0859778, longitude -70.78677  
**Search Strategy:** Drive-time radius (10 min, 12 min, 15 min)  
**Filter:** Car washes with >50,000 visits per year  
**Category:** Shops & Services > Car Wash Services > Car Wash

**Note:** POI search uses geographic radius, then filters by actual drive time using Google Routes API.

In [ ]:
from pydantic import BaseModel
from typing import Optional, List


class CategoryInfo(BaseModel):
    category: str
    group: str
    subCategory: str


class Address(BaseModel):
    city: str
    state: str
    countryCode: str
    streetName: str
    formattedAddress: str
    shortFormattedAddress: str
    zipCode: str


class RegionDetail(BaseModel):
    code: str
    name: str


class Regions(BaseModel):
    dma: RegionDetail
    state: RegionDetail
    cbsa: RegionDetail


class Venue(BaseModel):
    entityId: str
    entityType: str
    name: str
    categoryInfo: CategoryInfo
    address: Address
    isFlagged: bool
    regions: Regions
    apiId: str
    placerUrl: str
    storeId: Optional[str] = None
    isPermitted: bool


class PlacerResponse(BaseModel):
    data: List[Venue]
    requestId: str


print("✓ Pydantic models defined")

In [ ]:
# Step 1: Search for car wash competitors within geographic radius
# Note: Will filter by drive-time (10, 12, 15 min) in a later step
import requests

# Target POI coordinates
target_lat = 43.0859778
target_lng = -70.78677

# Search parameters
base_url = "https://papi.placer.ai/v1/poi"
headers = {
    "accept": "application/json",
    "x-api-key": f"{placer_api_key}"
}

params = {
    "lat": target_lat,
    "lng": target_lng,
    "radius": 5,  # Larger radius to capture all venues within 15 min drive time
    "category": "Car Wash Services",
    "subCategory": "Car Wash",
    "entityType": "venue",
    "limit": 50,
    "skip": 0
}

print(f"Searching for Car Wash competitors within {params['radius']} mile radius...\n")
print(f"(Will filter by drive-time in next step)\n")

response = requests.get(base_url, params=params, headers=headers)

if response.status_code == 200:
    result = PlacerResponse.model_validate_json(response.text)
    competitors = result.data
    
    print(f"Found {len(competitors)} car wash venues\n")
    print("Competitors found:")
    for venue in competitors:
        print(f"  - {venue.name} - {venue.address.formattedAddress}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)
    competitors = []

In [ ]:
# Step 1.5: Calculate drive times and filter by drive-time radius
# Uses Google Routes API to get actual drive times from target POI to each competitor
import googlemaps

# Initialize Google Maps client
gmaps = googlemaps.Client(key=google_api_key)

# Original POI coordinates
origin_lat = target_lat
origin_lng = target_lng

# Calculate drive times for each venue
venue_drive_times = {}

print("\n" + "="*80)
print("Calculating drive times from target POI (1313 Woodbury Ave)...")
print("="*80 + "\n")

for venue in competitors:
    address = venue.address.formattedAddress
    
    try:
        # Geocode to get destination coordinates
        geocode_result = gmaps.geocode(address)
        
        if geocode_result:
            venue_location = geocode_result[0]['geometry']['location']
            venue_lat = venue_location['lat']
            venue_lng = venue_location['lng']
            
            # Use Routes API to compute routes
            url = "https://routes.googleapis.com/directions/v2:computeRoutes"
            
            payload = {
                "origin": {
                    "location": {
                        "latLng": {
                            "latitude": origin_lat,
                            "longitude": origin_lng
                        }
                    }
                },
                "destination": {
                    "location": {
                        "latLng": {
                            "latitude": venue_lat,
                            "longitude": venue_lng
                        }
                    }
                },
                "travelMode": "DRIVE",
                "routingPreference": "TRAFFIC_AWARE",
                "computeAlternativeRoutes": False,
                "routeModifiers": {
                    "avoidTolls": False,
                    "avoidHighways": False,
                    "avoidFerries": False
                },
                "languageCode": "en-US",
                "units": "IMPERIAL"
            }
            
            headers_routes = {
                "Content-Type": "application/json",
                "X-Goog-Api-Key": google_api_key,
                "X-Goog-FieldMask": "routes.duration,routes.distanceMeters"
            }
            
            response = requests.post(url, json=payload, headers=headers_routes)
            
            if response.status_code == 200:
                result = response.json()
                
                if 'routes' in result and len(result['routes']) > 0:
                    route = result['routes'][0]
                    distance_meters = route['distanceMeters']
                    distance_miles = distance_meters * 0.000621371
                    duration_seconds = int(route['duration'].rstrip('s'))
                    duration_minutes = duration_seconds / 60
                    
                    venue_drive_times[venue.apiId] = {
                        'distance_miles': distance_miles,
                        'duration_minutes': duration_minutes,
                        'duration_text': f"{int(duration_minutes)} min"
                    }
                    
                    print(f"  {venue.name[:35]:<35} - {distance_miles:.2f} mi, {duration_minutes:.1f} min")
    except Exception as e:
        print(f"  {venue.name[:35]:<35} - Error: {str(e)[:40]}")

# Filter competitors by drive-time thresholds (10, 12, 15 min)
print(f"\n{'='*80}")
print("Filtering by drive-time thresholds...")
print(f"{'='*80}\n")

# Define drive-time bands
drive_time_bands = {
    '10min': 10,
    '12min': 12,
    '15min': 15
}

# Create filtered lists for each band
competitors_by_drive_time = {
    '10min': [],
    '12min': [],
    '15min': []
}

for venue in competitors:
    drive_info = venue_drive_times.get(venue.apiId)
    if drive_info:
        duration = drive_info['duration_minutes']
        
        # Assign to appropriate band
        if duration <= 10:
            competitors_by_drive_time['10min'].append(venue)
        elif duration <= 12:
            competitors_by_drive_time['12min'].append(venue)
        elif duration <= 15:
            competitors_by_drive_time['15min'].append(venue)

# Use the 15min band as the main competitors list (includes all within 15 min)
all_within_15min = []
for band in ['10min', '12min', '15min']:
    all_within_15min.extend(competitors_by_drive_time[band])

competitors = all_within_15min

print(f"  Within 10 min: {len(competitors_by_drive_time['10min'])} venues")
print(f"  Within 12 min: {len(competitors_by_drive_time['12min'])} venues (additional)")
print(f"  Within 15 min: {len(competitors_by_drive_time['15min'])} venues (additional)")
print(f"\n  Total within 15 min drive-time: {len(competitors)} venues")

print(f"\n{'='*80}")

In [ ]:
# Pydantic models for Visit Trends API response
from datetime import datetime, timedelta

class VisitTrendsData(BaseModel):
    visitDurationSegmentation: str
    apiId: str
    dates: List[str]
    visits: List[int]
    panelVisits: List[int]


class NoContentError(BaseModel):
    apiId: str
    details: str
    error: str
    code: int


class VisitTrendsResponse(BaseModel):
    data: List[VisitTrendsData | NoContentError]
    requestId: str


# Calculate date range (one year ago to 3 days ago)
end_date = (datetime.now() - timedelta(days=4)).strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

print(f"✓ Visit Trends models defined")
print(f"Date range: {start_date} to {end_date}")

In [ ]:
# Step 2: Get visit trends for all competitors to filter those with >50k visits/year
import time

url = "https://papi.placer.ai/v1/reports/visit-trends"

headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "x-api-key": placer_api_key
}

# Build apiIds list from competitors
api_ids = [venue.apiId for venue in competitors]

print(f"Fetching yearly visit data for {len(api_ids)} competitors...\n")

payload = {
    "apiIds": api_ids,
    "granularity": "month",  # Monthly data for yearly total
    "visitDurationSegmentation": "default",
    "startDate": start_date,
    "endDate": end_date
}

# Retry for IN_PROGRESS responses
visit_trends = []
for attempt in range(10):
    response = requests.post(url, json=payload, headers=headers)
    
    if response.status_code not in [200, 207]:
        print(f"✗ HTTP {response.status_code}")
        break
    
    if "IN_PROGRESS" in response.text:
        print(f"  Attempt {attempt + 1}: Data processing...", end="")
        time.sleep(3)
        continue
    
    try:
        trends_response = VisitTrendsResponse.model_validate_json(response.text)
        
        for item in trends_response.data:
            if isinstance(item, VisitTrendsData):
                visit_trends.append(item)
        
        print(f"✓ Got visit data for {len(visit_trends)} venues")
        break
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        break

# Filter competitors with >50k visits per year
filtered_competitors = []
competitor_visit_data = {}

print(f"\nFiltering for competitors with >50,000 visits per year:\n")

for trend in visit_trends:
    yearly_total = sum(trend.visits)
    
    if yearly_total > 50000:
        # Find the venue
        venue = next((v for v in competitors if v.apiId == trend.apiId), None)
        if venue:
            filtered_competitors.append(venue)
            competitor_visit_data[venue.apiId] = {
                'yearly_total': yearly_total,
                'monthly_avg': yearly_total // 12,
                'daily_avg': yearly_total // 365
            }
            print(f"  ✓ {venue.name}: {yearly_total:,} visits/year")

print(f"\n{'='*80}")
print(f"Qualified competitors: {len(filtered_competitors)} (out of {len(competitors)} total)")
print(f"{'='*80}")

# Update competitors list to only include filtered ones
competitors = filtered_competitors

In [ ]:
# Pydantic models for Comparison API (Loyalty & Trade Area)
from typing import Dict, Any

class LoyaltyFrequencyData(BaseModel):
    apiId: str
    startDate: str
    endDate: str
    avgVisitsPerCustomer: float
    medianVisitsPerCustomer: int
    bins: List[int]
    visitors: List[int]
    visitorsPercentage: List[float]
    visits: List[int]
    visitsPercentage: List[float]
    visitDurationSegmentation: str


class LoyaltyFrequencyResponse(BaseModel):
    data: LoyaltyFrequencyData
    requestId: str


class TradeAreaData(BaseModel):
    type: str  # "MultiPolygon"
    coordinates: List[List[List[List[float]]]]  # GeoJSON MultiPolygon coordinates
    visitDurationSegmentation: str


class TradeAreaResponse(BaseModel):
    apiId: str
    data: TradeAreaData


print("✓ Comparison API models defined")

In [ ]:
# Step 3: Fetch loyalty data (visitors with 13+ visits = "Total Members")
# Uses the loyalty/visits-frequency endpoint to get segmentation by visit frequency

url = "https://papi.placer.ai/v1/reports/loyalty/visits-frequency"

headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "x-api-key": placer_api_key
}

print(f"Fetching loyalty data for {len(competitors)} competitors...\n")

loyalty_data = {}

# Call endpoint for each competitor individually (not batched)
for competitor in competitors:
    api_id = competitor.apiId
    
    payload = {
        "apiId": api_id,
        "startDate": start_date,
        "endDate": end_date
    }
    
    for attempt in range(10):
        response = requests.post(url, json=payload, headers=headers)
        
        # Check for IN_PROGRESS first (can be 202 or 200 with IN_PROGRESS in body)
        if "IN_PROGRESS" in response.text or response.status_code == 202:
            if attempt == 0:
                print(f"  {competitor.name}: Processing...", end="")
            else:
                print(".", end="")
            time.sleep(3)
            continue
        
        # Now check for error status codes
        if response.status_code not in [200, 207]:
            print(f"\n  ✗ {competitor.name}: HTTP {response.status_code}")
            print(f"     {response.text[:150]}")
            break
        
        try:
            # Parse response with Pydantic model
            result = LoyaltyFrequencyResponse.model_validate_json(response.text)
            
            # Calculate loyal visitors (13+ visits)
            # bins example: [1,2,3,4,5,6,7,8,9,10,15,20,30]
            # We need to sum visitors for bins >= 13
            loyal_visitors = 0
            
            for i, bin_value in enumerate(result.data.bins):
                if bin_value >= 13 and i < len(result.data.visitors):
                    loyal_visitors += result.data.visitors[i]
            
            loyalty_data[api_id] = {
                'total_members': loyal_visitors,
                'avg_visits_per_customer': result.data.avgVisitsPerCustomer,
                'median_visits': result.data.medianVisitsPerCustomer
            }
            
            print(f"\n  ✓ {competitor.name}: {loyal_visitors:,} loyal members (13+ visits)")
            break
            
        except Exception as e:
            print(f"\n  ✗ {competitor.name}: Error - {str(e)}")
            print(f"     Response: {response.text[:300]}")
            break
    
    time.sleep(0.5)  # Rate limiting between requests

print(f"\n{'='*80}")
print(f"Loyalty data collected for {len(loyalty_data)} competitors")
print(f"{'='*80}")

In [ ]:
# Step 4a: Find ANY nearby business to use as reference for trade area comparison
# Since we're analyzing land for a NEW car wash, we need a nearby business to represent
# what the trade area would look like at this location
import json

print("Finding a nearby business to use as trade area reference...\n")
print("(Does not need to be a car wash - just needs to represent the target location)\n")

# Search for ANY venue near the target location
target_search_url = "https://papi.placer.ai/v1/poi"
target_params = {
    "lat": target_lat,
    "lng": target_lng,
    "radius": 0.5,  # Start with 0.5 miles
    "entityType": "venue",
    "limit": 20
}

print(f"Searching for businesses near target location...\n")
print(f"Search parameters: lat={target_lat}, lng={target_lng}, radius=0.5 miles\n")

response = requests.get(target_search_url, params=target_params, headers={
    "accept": "application/json",
    "x-api-key": placer_api_key
})

print(f"Response status: {response.status_code}\n")

target_api_id = None
target_name = None

if response.status_code == 200:
    result = response.json()
    
    if result.get('data'):
        # Parse with Pydantic
        parsed_result = PlacerResponse.model_validate_json(response.text)
        
        print(f"Found {len(parsed_result.data)} nearby businesses:\n")
        print("="*80)
        
        for i, venue in enumerate(parsed_result.data[:10], 1):  # Show first 10
            print(f"{i}. {venue.name}")
            print(f"   Category: {venue.categoryInfo.category}")
            print(f"   Address: {venue.address.formattedAddress}")
            print(f"   API ID: {venue.apiId}")
            print()
        
        # Use the closest one (first in results) as reference
        if parsed_result.data:
            target_api_id = parsed_result.data[0].apiId
            target_name = parsed_result.data[0].name
            target_category = parsed_result.data[0].categoryInfo.category
            print("="*80)
            print(f"\n✓ Using as reference POI: {target_name}")
            print(f"  Category: {target_category}")
            print(f"  API ID: {target_api_id}\n")
        else:
            print("⚠ No businesses found nearby. Will skip trade area overlap analysis.\n")
    else:
        print("⚠ No data returned. Will skip trade area overlap analysis.\n")
else:
    print(f"Error: {response.text}")

if not target_api_id:
    print("\n⚠️  Note: If no nearby business found, trade area overlap will be skipped.")
    print("   Consider widening the search radius or using placeholder values.")

In [ ]:
# Step 4: Fetch trade areas and calculate overlap percentages
from shapely.geometry import shape, MultiPolygon
from shapely.ops import unary_union

print("Fetching trade area polygons for overlap calculation...\n")

# Fetch reference POI trade area
print(f"Fetching reference POI trade area...")

trade_area_url = "https://papi.placer.ai/v1/reports/true-trade-area"

headers_trade = {
    "accept": "application/json",
    "content-type": "application/json",
    "x-api-key": placer_api_key
}

payload = {
    "apiId": target_api_id,
    "startDate": start_date,
    "endDate": end_date,
    "tradeAreaType": "70percentTrueTradeArea"
}

reference_polygon = None

for attempt in range(10):
    response = requests.post(trade_area_url, json=payload, headers=headers_trade)
    
    if "IN_PROGRESS" in response.text or response.status_code == 202:
        print(".", end="")
        time.sleep(3)
        continue
    
    if response.status_code == 200:
        result = TradeAreaResponse.model_validate_json(response.text)
        # Convert GeoJSON to shapely geometry
        geojson = {"type": result.data.type, "coordinates": result.data.coordinates}
        reference_polygon = shape(geojson)
        print(f" ✓ Reference trade area loaded")
        break
    else:
        print(f" ✗ Error: {response.status_code}")
        print(f"Response: {response.text}")
        break

# Fetch trade areas for all competitors and calculate overlap
print(f"\nCalculating trade area overlaps for {len(competitors)} competitors...\n")

trade_area_data = {}

for competitor in competitors:
    payload = {
        "apiId": competitor.apiId,
        "startDate": start_date,
        "endDate": end_date,
        "tradeAreaType": "70percentTrueTradeArea"
    }
    
    for attempt in range(10):
        response = requests.post(trade_area_url, json=payload, headers=headers_trade)
        
        if "IN_PROGRESS" in response.text or response.status_code == 202:
            if attempt == 0:
                print(f"  {competitor.name}: Processing...", end="")
            else:
                print(".", end="")
            time.sleep(3)
            continue
        
        if response.status_code == 200:
            result = TradeAreaResponse.model_validate_json(response.text)
            
            # Convert GeoJSON to shapely geometry
            geojson = {"type": result.data.type, "coordinates": result.data.coordinates}
            competitor_polygon = shape(geojson)
            
            # Calculate overlap percentage
            if reference_polygon and competitor_polygon.is_valid and reference_polygon.is_valid:
                try:
                    # Calculate intersection
                    intersection = reference_polygon.intersection(competitor_polygon)
                    
                    # Calculate overlap as percentage of competitor's trade area
                    competitor_area = competitor_polygon.area
                    intersection_area = intersection.area
                    
                    overlap_pct = (intersection_area / competitor_area * 100) if competitor_area > 0 else 0
                    
                    # Store both overlap percentage AND polygon geometry for later use
                    trade_area_data[competitor.apiId] = {
                        'overlap_percentage': overlap_pct,
                        'polygon': competitor_polygon  # Store for visualization reuse
                    }
                    
                    print(f"\n  ✓ {competitor.name}: {overlap_pct:.1f}% overlap")
                except Exception as e:
                    print(f"\n  ✗ {competitor.name}: Calculation error - {str(e)[:50]}")
            else:
                print(f"\n  ⚠ {competitor.name}: Invalid polygon geometry")
            break
        else:
            print(f"\n  ✗ {competitor.name}: HTTP {response.status_code}")
            print(f"     Response: {response.text}")
            break
    
    time.sleep(0.5)  # Rate limiting

print(f"\n{'='*80}")
print(f"Trade area overlap calculated for {len(trade_area_data)} competitors")
print(f"{'='*80}")

In [ ]:
# Step 5: Calculate derived metrics

# Calculate Members in Market and Share of Target for each competitor
competitor_metrics = {}

for competitor in competitors:
    api_id = competitor.apiId
    
    # Get base data
    total_members = loyalty_data.get(api_id, {}).get('total_members', 0)
    overlap_pct = trade_area_data.get(api_id, {}).get('overlap_percentage', 0)
    
    # Calculate derived metrics
    members_in_market = (overlap_pct / 100) * total_members
    
    competitor_metrics[api_id] = {
        'total_members': total_members,
        'overlap_percentage': overlap_pct,
        'members_in_market': members_in_market,
    }

print("✓ Derived metrics calculated for all competitors")

In [ ]:
# Step 6: Display comprehensive competitor data

print(f"\n{'='*200}")
print(f"COMPETITOR ANALYSIS - CAR WASH SERVICES")
print(f"Target POI: 1313 Woodbury Ave, Portsmouth, NH")
print(f"Drive-Time Bands: 0-10 min, 10-12 min, 12-15 min | Filter: >50,000 visits/year")
print(f"{'='*200}\n")

print(f"{'Name':<30} {'Drive':<8} {'Type':<12} {'Quality':<10} {'Total Members':<15} {'% Overlap':<12} {'Members Mkt':<14} {'Visits/Year':<15}")
print(f"{'='*200}")

for competitor in competitors:
    api_id = competitor.apiId
    
    # Base info
    name = competitor.name[:29]
    
    # Drive time info
    drive_info = venue_drive_times.get(api_id, {})
    drive_time = drive_info.get('duration_text', 'N/A')
    
    # Type and Quality - To be populated from template (placeholder for now)
    comp_type = "TBD"  # Will be dropdown from template
    quality = "TBD"    # Will be dropdown from template
    
    # Metrics
    metrics = competitor_metrics.get(api_id, {})
    total_members = metrics.get('total_members', 0)
    overlap_pct = metrics.get('overlap_percentage', 0)
    members_in_market = metrics.get('members_in_market', 0)
    
    # Visit data
    visit_data = competitor_visit_data.get(api_id, {})
    yearly_visits = visit_data.get('yearly_total', 0)
    
    print(f"{name:<30} {drive_time:<8} {comp_type:<12} {quality:<10} {total_members:>14,} {overlap_pct:>10.1f}% {members_in_market:>13,.0f} {yearly_visits:>14,}")

print(f"{'='*200}")
print(f"\nTotal Competitors: {len(competitors)}")
print(f"\nNote: 'Type' and 'Quality' fields need to be populated from template dropdowns")

In [ ]:
# Step 7: Export data to structured format
# Note: Drive-time bands: 10min, 12min, 15min

print("\n" + "="*180)
print("COMPETITOR DATA EXPORT")
print("="*180 + "\n")

# Build export data structure
export_data = []

for competitor in competitors:
    api_id = competitor.apiId
    metrics = competitor_metrics.get(api_id, {})
    visit_data = competitor_visit_data.get(api_id, {})
    drive_info = venue_drive_times.get(api_id, {})
    
    # Determine drive-time band
    duration_min = drive_info.get('duration_minutes', 999)
    if duration_min <= 10:
        drive_time_band = "0-10 min"
    elif duration_min <= 12:
        drive_time_band = "10-12 min"
    elif duration_min <= 15:
        drive_time_band = "12-15 min"
    else:
        drive_time_band = "15+ min"
    
    export_data.append({
        'Name': competitor.name,
        'Address': competitor.address.formattedAddress,
        'Drive Time': drive_info.get('duration_text', 'N/A'),
        'Drive Time Band': drive_time_band,
        'Distance (mi)': round(drive_info.get('distance_miles', 0), 2),
        'Type': 'TBD',  # To be populated from template
        'Quality': 'TBD',  # To be populated from template
        'Total Members': metrics.get('total_members', 0),
        '% Overlap': metrics.get('overlap_percentage', 0),
        'Members in Market': int(metrics.get('members_in_market', 0)),
        'Visits per Year': visit_data.get('yearly_total', 0),
        'Visits per Month (Avg)': visit_data.get('monthly_avg', 0),
        'Visits per Day (Avg)': visit_data.get('daily_avg', 0),
        'API ID': api_id
    })

# Sort by Members in Market descending
export_data.sort(key=lambda x: x['Members in Market'], reverse=True)

# Display data in formatted table
header = "| {:<30} | {:<12} | {:<15} | {:<12} | {:<15} | {:<12} | {:<16} | {:<15} |".format(
    "Name", "Drive Time", "Drive Band", "Total Members", "% Overlap", "Members Mkt", "Visits/Year", "Distance (mi)"
)
separator = "-" * len(header)

print(header)
print(separator)

for item in export_data:
    row = "| {:<30} | {:<12} | {:<15} | {:<12,} | {:<12.1f}% | {:<16,} | {:<15,} | {:<15.2f} |".format(
        item['Name'][:29],
        item['Drive Time'],
        item['Drive Time Band'],
        item['Total Members'],
        item['% Overlap'],
        item['Members in Market'],
        item['Visits per Year'],
        item['Distance (mi)']
    )
    print(row)

print(separator)
print(f"\nTotal Competitors: {len(export_data)}")

# Summary by drive-time band
print(f"\n{'='*80}")
print("Summary by Drive-Time Band:")
print(f"{'='*80}")

bands = {}
for item in export_data:
    band = item['Drive Time Band']
    if band not in bands:
        bands[band] = 0
    bands[band] += 1

for band in sorted(bands.keys()):
    print(f"  {band}: {bands[band]} competitors")

print(f"\n✓ Data export complete (without pandas)")

In [ ]:
# Step 4c: Visualize trade area overlaps using Google Maps
import sys
sys.path.append('/Users/scottkellar/Work/grholdings-carwash-api')

from src.services.google_location_service import GoogleLocationService
from src.models.address import Coordinates
from IPython.display import Image, display
import requests

print("Creating satellite visualization with trade area overlaps...\n")

def add_single_polygon_overlay(base_url, polygon, color):
    """
    Add a single simplified polygon overlay to Google Static Maps URL
    """
    url = base_url
    
    # Very aggressive simplification to keep URL short
    simplified_polygon = polygon.simplify(0.02, preserve_topology=True)
    
    # Convert to coordinates
    if simplified_polygon.geom_type == 'MultiPolygon':
        geojson = simplified_polygon.__geo_interface__
        coords = geojson['coordinates']
    elif simplified_polygon.geom_type == 'Polygon':
        coords = [[list(simplified_polygon.exterior.coords)]]
    else:
        return url
    
    # Only use the largest polygon from MultiPolygon
    if len(coords) > 1:
        # Find largest polygon by number of points
        coords = [max(coords, key=lambda x: len(x[0]))]
    
    # Take only the first ring (exterior)
    for polygon_set in coords[:1]:
        for polygon_ring in polygon_set[:1]:
            # Ultra-simplified - max 15 points
            step = max(1, len(polygon_ring) // 15)
            simplified = polygon_ring[::step]
            
            # Format path
            path_points = "|".join([f"{lat},{lng}" for lng, lat in simplified])
            url += f"&path=color:{color}|weight:3|fillcolor:{color}77|{path_points}"
    
    return url

def add_markers_to_map_url(base_url, markers):
    """
    Add markers to Google Static Maps URL with larger size
    markers: list of dicts with 'lat', 'lng', 'color', 'label'
    """
    url = base_url
    
    for marker in markers:
        # Format: &markers=size:mid|color:blue|label:T|lat,lng
        color = marker.get('color', 'red')
        label = marker.get('label', '')
        lat = marker['lat']
        lng = marker['lng']
        size = marker.get('size', 'mid')  # Use mid-size markers for better visibility
        
        if label:
            url += f"&markers=size:{size}|color:{color}|label:{label}|{lat},{lng}"
        else:
            url += f"&markers=size:{size}|color:{color}|{lat},{lng}"
    
    return url

# Initialize Google Location Service
location_service = GoogleLocationService(api_key=google_api_key, image_size="900x600")

if reference_polygon and target_api_id and len(competitors) > 0:
    coords = Coordinates(
        latitude=target_lat,
        longitude=target_lng,
        formatted_address=f"{target_lat},{target_lng}"
    )
    
    # Get base satellite map URL (zoom 13 for closer view)
    base_url = location_service.get_satellite_url(coords, satellite_zoom_level=12)
    
    print(f"Using cached competitor trade area (from Step 4)...\n")
    
    competitor = competitors[0]
    
    # Reuse the polygon we already fetched in Step 4 (no duplicate API call!)
    competitor_polygon = trade_area_data.get(competitor.apiId, {}).get('polygon')
    
    if competitor_polygon:
        print(f"✓ Retrieved polygon for {competitor.name}\n")
        
        # Calculate intersection
        intersection = reference_polygon.intersection(competitor_polygon)
        
        # Calculate overlap percentage
        overlap_pct = (intersection.area / competitor_polygon.area * 100) if competitor_polygon.area > 0 else 0
        
        print("="*80)
        print("TRADE AREA OVERLAP VISUALIZATION")
        print("="*80)
        print(f"\nShowing overlap area (green) on satellite imagery")
        print(f"  Reference: {target_name} (Market Basket)")
        print(f"  Competitor: {competitor.name}")
        print(f"  Overlap: {overlap_pct:.1f}% of competitor's trade area")
        print(f"  Zoom level: 13")
        print(f"\n{'='*80}\n")
        
        # Create map with polygon overlay
        map_url = add_single_polygon_overlay(base_url, intersection, '0x00ff00')
        
        # Add markers for target POI and competitors with LARGER SIZE
        markers = [
            {
                'lat': target_lat,
                'lng': target_lng,
                'color': 'blue',
                'label': 'T',
                'size': 'mid'  # Larger marker
            }
        ]
        
        # Add competitor markers
        gmaps = googlemaps.Client(key=google_api_key)
        for i, comp in enumerate(competitors[:5]):  # Max 5 to keep URL short
            drive_info = venue_drive_times.get(comp.apiId, {})
            # Geocode competitor address to get coordinates
            try:
                geocode_result = gmaps.geocode(comp.address.formattedAddress)
                if geocode_result:
                    comp_lat = geocode_result[0]['geometry']['location']['lat']
                    comp_lng = geocode_result[0]['geometry']['location']['lng']
                    markers.append({
                        'lat': comp_lat,
                        'lng': comp_lng,
                        'color': 'red',
                        'label': str(i+1),
                        'size': 'mid'  # Larger marker
                    })
            except:
                pass
        
        # Add markers to map
        map_url = add_markers_to_map_url(map_url, markers)
        
        print(f"Map URL length: {len(map_url)} characters\n")
        print("Map Legend:")
        print("  🔵 T = Target POI (1313 Woodbury Ave) - BLUE MARKER")
        for i, comp in enumerate(competitors[:5]):
            drive_info = venue_drive_times.get(comp.apiId, {})
            drive_time = drive_info.get('duration_text', 'N/A')
            print(f"  🔴 {i+1} = {comp.name} ({drive_time}) - RED MARKER")
        print("  🟢 GREEN AREA = Trade Area Overlap")
        print("\nNote: Markers are mid-size. Look for blue 'T' and red numbers.\n")
        
        # Display the image
        try:
            response = requests.get(map_url)
            if response.status_code == 200:
                display(Image(response.content))
            else:
                print(f"⚠ Could not fetch map: HTTP {response.status_code}")
                if response.status_code == 413:
                    print("  URL still too long. Polygon may be too complex.")
        except Exception as e:
            print(f"⚠ Error: {str(e)}")
    else:
        print("⚠ Could not retrieve competitor polygon from Step 4")
else:
    print("⚠ Missing required data for visualization")

In [ ]:
# Step 8: Competitor Summary Table (Formatted like template)

print("\n" + "="*120)
print("COMPETITOR SUMMARY - CAR WASH MARKET ANALYSIS")
print("="*120 + "\n")

# Calculate Total Addressable Market (TAM) based on car parc
# Using 15 min drive time since both competitors fall in the 12-15 min band
# TAM = Total Car Parc × Total Addressable Members %
car_parc_15min = 58_630
tam_percentage_15min = 20  # 20% conversion rate for 15 min drive time
total_addressable_market = int(car_parc_15min * tam_percentage_15min / 100)

print(f"Total Addressable Market (TAM): {total_addressable_market:,} members")
print(f"  (Calculated from: {car_parc_15min:,} car parc × {tam_percentage_15min}% conversion rate at 15 min drive time)\n")

# Display competitor table
print(f"{'Competitors':<30} {'Type':<15} {'Quality':<12} {'Total Members':>15} {'% Overlap':>12} {'Members in Market':>18}")
print("-" * 120)

total_members_in_market = 0

for competitor in competitors:
    api_id = competitor.apiId
    name = competitor.name[:29]
    
    # Type and Quality - placeholder values (to be populated from template/manual entry)
    comp_type = "Express" if "Envi" in competitor.name else "IBA/Flex"
    quality = "Weak" if "Envi" in competitor.name else "Terrible"
    
    # Get metrics
    metrics = competitor_metrics.get(api_id, {})
    total_members = metrics.get('total_members', 0)
    overlap_pct = metrics.get('overlap_percentage', 0)
    members_in_market = int(metrics.get('members_in_market', 0))
    
    total_members_in_market += members_in_market
    
    print(f"{name:<30} {comp_type:<15} {quality:<12} {total_members:>15,} {overlap_pct:>11.0f}% {members_in_market:>18,}")

print("-" * 120)
print(f"{'Total Market Members':<30} {'':<15} {'':<12} {'':<15} {'':<12} {total_members_in_market:>18,}")

# Calculate Current Share of Target
# This represents what % of the total addressable market is already captured by competitors
# Formula: Current Share of Target = Total Market Members / TAM × 100
current_share_pct = (total_members_in_market / total_addressable_market * 100) if total_addressable_market > 0 else 0

print(f"{'Current Share of Target (>50% = Market Share Battle)':<59} {current_share_pct:>18.0f}%")

